# Test d'encodage EmbeddingGemma 2 (expérience 1, étape 5a)

But : mesurer, sur 500 essais tirés au hasard, (1) la longueur des textes en tokens selon ce qu'on encode, (2) la vitesse d'encodage sur le GPU T4, (3) l'accord entre float32 et bfloat16. Aucune donnée n'est gardée sur Colab.

**Avant de lancer** : menu *Exécution > Modifier le type d'exécution > GPU T4*. Puis *Exécution > Tout exécuter*. La cellule 3 demande le fichier `embedding_sample_500.json.gz` (dans `data/processed/` du projet). À la fin, le fichier `embedding_test_result.json` se télécharge : il faut le remettre dans `results/` du projet.

In [ ]:
# 1. Vérifier le GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Pas de GPU : choisir GPU T4 dans Exécution > Modifier le type d'exécution"
print("torch", torch.__version__, "| bf16 annoncé :", torch.cuda.is_bf16_supported())

In [ ]:
# 2. Installer les bibliothèques (versions affichées pour la reproductibilité)
!pip install -q -U "sentence-transformers>=6.1" transformers
import sentence_transformers, transformers
print("sentence-transformers", sentence_transformers.__version__, "| transformers", transformers.__version__)

In [ ]:
# 3. Envoyer l'échantillon (data/processed/embedding_sample_500.json.gz)
import gzip, json
from google.colab import files
uploaded = files.upload()
sample = json.loads(gzip.decompress(uploaded["embedding_sample_500.json.gz"]))
print(len(sample), "essais")

In [ ]:
# 4. Charger le modèle : texte seul, révision figée, float32 (la fiche interdit float16)
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer
MODEL = "google/embeddinggemma-2"
REVISION = HfApi().model_info(MODEL).sha
print("révision figée :", REVISION)
def load(dtype):
    model = SentenceTransformer(MODEL, revision=REVISION, device="cuda",
                                model_kwargs={"torch_dtype": dtype},
                                config_kwargs={"vision_config": None, "audio_config": None})
    return model
model = load(torch.float32)
print("longueur max :", model.max_seq_length)

In [ ]:
# 5. Deux façons de composer le texte d'un essai (format document de la fiche : « title: ... | text: ... »)
import numpy as np
def compose(trial, variant):
    parts = ["; ".join(trial["conditions"])]
    if variant == "complet":
        parts += [trial["summary"], trial["detailed_description"]]
    parts.append(trial["eligibility"])
    body = " ".join(" ".join(p.split()) for p in parts if p)
    return f"title: {trial['title'] or 'none'} | text: {body}"
VARIANTS = ("coeur", "complet")
texts = {v: [compose(t, v) for t in sample] for v in VARIANTS}
result = {"model": MODEL, "revision": REVISION, "n_trials": len(sample), "gpu": torch.cuda.get_device_name(0),
          "versions": {"torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__, "transformers": transformers.__version__},
          "variants": {"coeur": "titre + conditions + critères", "complet": "titre + conditions + résumé + description + critères"},
          "tokens": {}, "speed": {}}
for v in VARIANTS:
    lengths = np.array([len(model.tokenizer(t)["input_ids"]) for t in texts[v]])
    result["tokens"][v] = {"median": int(np.median(lengths)), "p90": int(np.percentile(lengths, 90)), "p99": int(np.percentile(lengths, 99)),
                           "max": int(lengths.max()), "over_limit": int((lengths > model.max_seq_length).sum())}
    print(v, result["tokens"][v])

In [ ]:
# 6. Vitesse d'encodage, float32 puis bfloat16 ; on réduit le lot si la mémoire manque
import time
def encode(m, docs, batch_size=16):
    while True:
        try:
            torch.cuda.synchronize(); start = time.perf_counter()
            emb = m.encode(docs, batch_size=batch_size, normalize_embeddings=True, convert_to_numpy=True)
            torch.cuda.synchronize(); return emb, time.perf_counter() - start, batch_size
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache(); batch_size //= 2
            if batch_size == 0: raise
embeddings = {}
for dtype_name, dtype in (("float32", torch.float32), ("bfloat16", torch.bfloat16)):
    m = model if dtype_name == "float32" else load(dtype)
    encode(m, texts["coeur"][:16])  # chauffe
    for v in VARIANTS:
        emb, seconds, batch = encode(m, texts[v])
        embeddings[(dtype_name, v)] = emb
        result["speed"][f"{dtype_name}/{v}"] = {"seconds": round(seconds, 1), "trials_per_second": round(len(texts[v]) / seconds, 1),
            "batch_size": batch, "nan": int(np.isnan(emb).sum()), "hours_for_375580": round(375580 / (len(texts[v]) / seconds) / 3600, 2)}
        print(dtype_name, v, result["speed"][f"{dtype_name}/{v}"])
    if m is not model: del m; torch.cuda.empty_cache()

In [ ]:
# 7. Accord bfloat16 / float32 (cosinus entre les deux vecteurs d'un même essai) et téléchargement du résultat
result["bf16_vs_fp32_cosine"] = {}
for v in VARIANTS:
    cos = (embeddings[("float32", v)] * embeddings[("bfloat16", v)]).sum(axis=1)
    result["bf16_vs_fp32_cosine"][v] = {"min": round(float(cos.min()), 5), "mean": round(float(cos.mean()), 5)}
print(json.dumps(result, indent=1, ensure_ascii=False))
with open("embedding_test_result.json", "w") as f: json.dump(result, f, indent=1, ensure_ascii=False)
files.download("embedding_test_result.json")